# HAM10000 — Clasificación final: synthetic-only y balanced con LoRA/GAN

Notebook limpio de análisis final. Lee resultados previos del mismo `EXP_ROOT` y corre 5 escenarios nuevos.

| Escenario | Train mel | Pregunta |
|---|---|---|
| `synthetic_only_lora` | 801 LoRA | ¿LoRA solo supera TI solo (AUC=0.686)? |
| `synthetic_only_gan` | 801 GAN-final | ¿GAN solo funciona mejor que TI solo? |
| `synthetic_only_mix` | 401 LoRA + 400 GAN-final (seed=29) | ¿Mezcla supera fuente única? |
| `real_balanced_lora` | 801 real + ~3874 LoRA (mel≈nv) | ¿LoRA en volumen supera real_balanced? |
| `real_balanced_gan` | 801 real + ~3874 GAN-final (mel≈nv) | ¿GAN en volumen supera real_balanced? |

**Prerequisitos Drive:** `classification_data.zip`, `synthetic/lora/`, `gan_final.zip`

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import torch

def get_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'CUDA: {name}  ({vram:.1f} GB)')
        return torch.device('cuda'), vram
    if torch.backends.mps.is_available():
        print('Apple MPS')
        return torch.device('mps'), 0
    print('CPU')
    return torch.device('cpu'), 0

DEVICE, VRAM_GB = get_device()
print(f'IN_COLAB={IN_COLAB}  device={DEVICE}')

In [ ]:
from pathlib import Path
import zipfile, struct, shutil, json, subprocess, sys

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT    = Path('/content/drive/MyDrive/ham10000-augmentation')
    ZIP_PATH      = DRIVE_ROOT / 'classification_data.zip'
    IMAGES_DIR    = Path('/content/images')
    SPLITS_DIR    = Path('/content/splits')
    SYNTH_ROOT    = DRIVE_ROOT / 'synthetic'
    GAN_FINAL_ZIP = next(DRIVE_ROOT.glob('gan_final*.zip'), None)
    GAN_FINAL_DIR = Path('/content/gan_final')
    EXP_ROOT      = DRIVE_ROOT / 'experiments'
else:
    PROJECT_ROOT  = Path.cwd()
    ZIP_PATH      = PROJECT_ROOT / 'data/processed/classification_data.zip'
    IMAGES_DIR    = PROJECT_ROOT / 'data/processed/images'
    SPLITS_DIR    = PROJECT_ROOT / 'data/processed/splits'
    SYNTH_ROOT    = PROJECT_ROOT / 'data/synthetic'
    GAN_FINAL_ZIP = next(PROJECT_ROOT.glob('gan_final*.zip'), None)
    GAN_FINAL_DIR = PROJECT_ROOT / 'data/synthetic/gan_final'
    EXP_ROOT      = PROJECT_ROOT / 'experiments'

LORA_DIR = SYNTH_ROOT / 'lora'
EXP_ROOT.mkdir(parents=True, exist_ok=True)

print(f'SYNTH_ROOT:    {SYNTH_ROOT}')
print(f'GAN_FINAL_DIR: {GAN_FINAL_DIR}')
print(f'LORA_DIR:      {LORA_DIR}')
print(f'EXP_ROOT:      {EXP_ROOT}')

In [ ]:
# ── Dependencias ─────────────────────────────────────────────────────────────
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import timm; print(f'timm {timm.__version__}')
except ImportError:
    pip('timm'); import timm
try:
    import sklearn; print(f'sklearn {sklearn.__version__}')
except ImportError:
    pip('scikit-learn'); import sklearn
try:
    import cv2
except ImportError:
    pip('opencv-python-headless'); import cv2
print('Dependencias listas')

In [ ]:
# ── Extraer imágenes reales + splits ─────────────────────────────────────────
if IN_COLAB:
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(IMAGES_DIR.glob('*.jpg'))) < 100:
        print('Extrayendo classification_data.zip...')
        with zipfile.ZipFile(ZIP_PATH) as zf:
            for m in zf.infolist():
                data = zf.read(m.filename)
                if m.filename.startswith('images/') and m.filename.endswith('.jpg'):
                    (IMAGES_DIR / Path(m.filename).name).write_bytes(data)
                elif m.filename.startswith('splits/') and m.filename.endswith('.csv'):
                    (SPLITS_DIR / Path(m.filename).name).write_bytes(data)
        print(f'  {len(list(IMAGES_DIR.glob("*.jpg")))} imágenes  |  splits listos')
    else:
        print(f'Imágenes ya extraídas ({len(list(IMAGES_DIR.glob("*.jpg")))})')

In [ ]:
# ── Copiar sintéticas de Drive a /content/ ───────────────────────────────────
if IN_COLAB:
    LOCAL_SYNTH = Path('/content/synthetic')
    if not LOCAL_SYNTH.exists():
        print('Copiando synthetic/ de Drive...')
        shutil.copytree(str(SYNTH_ROOT), str(LOCAL_SYNTH))
        print(f'  {len(list(LOCAL_SYNTH.glob("**/*.jpg")))} imágenes')
    else:
        print(f'Sintéticas ya en local')
    SYNTH_ROOT = LOCAL_SYNTH
    LORA_DIR   = SYNTH_ROOT / 'lora'

# ── Extraer GAN-final ────────────────────────────────────────────────────────
if IN_COLAB:
    GAN_FINAL_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(GAN_FINAL_DIR.glob('*.png'))) < 10:
        assert GAN_FINAL_ZIP and GAN_FINAL_ZIP.exists(), 'Sube gan_final.zip a Drive'
        print(f'Extrayendo {GAN_FINAL_ZIP.name}...')
        with zipfile.ZipFile(GAN_FINAL_ZIP) as zf:
            for m in zf.infolist():
                if m.filename.endswith('.png'):
                    (GAN_FINAL_DIR / Path(m.filename).name).write_bytes(zf.read(m.filename))

n_lora  = len(list(LORA_DIR.glob('*.jpg'))) if LORA_DIR.exists() else 0
n_gan   = len(list(GAN_FINAL_DIR.glob('*.png'))) if GAN_FINAL_DIR.exists() else 0
print(f'LoRA: {n_lora}  |  GAN-final: {n_gan}')

In [ ]:
# ── Estado: todos los experimentos (previos + nuevos) ────────────────────────
ALL_PREV = ['real_only','real_balanced','real_2x','synthetic_only',
            'img2img_2x','ti_filtered_2x','gan64_2x','img2img_all','img2img_only',
            'gan_final_2x','lora_2x']
NEW_SCENARIOS = ['synthetic_only_lora','synthetic_only_gan','synthetic_only_mix',
                 'real_balanced_lora','real_balanced_gan']

def find_run_dir(sc):
    pointer = EXP_ROOT / f'{sc}_current.txt'
    if pointer.exists():
        return EXP_ROOT / pointer.read_text().strip()
    completed = sorted([d for d in EXP_ROOT.glob(f'*_{sc}')
                        if (d / 'test_metrics.json').exists()])
    return completed[-1] if completed else None

print('── Experimentos previos ──')
for sc in ALL_PREV:
    run_dir = find_run_dir(sc)
    if run_dir and (run_dir / 'test_metrics.json').exists():
        m = json.loads((run_dir / 'test_metrics.json').read_text())
        print(f'  ✅ {sc:25s}  AUC={m["auc"]}  Recall={m["recall_mel"]}  F1={m["f1_mel"]}')
    else:
        print(f'  ⬜ {sc}')

print('\n── Escenarios nuevos ──')
for sc in NEW_SCENARIOS:
    run_dir = find_run_dir(sc)
    if run_dir and (run_dir / 'test_metrics.json').exists():
        m = json.loads((run_dir / 'test_metrics.json').read_text())
        print(f'  ✅ {sc:25s}  AUC={m["auc"]}  Recall={m["recall_mel"]}  F1={m["f1_mel"]}')
    elif run_dir:
        ckpt = run_dir / 'checkpoint_last.pt'
        print(f'  🔄 {sc:25s}  {"retomable" if ckpt.exists() else "iniciado"}')
    else:
        print(f'  ⬜ {sc}')

In [ ]:
# ── Hiperparámetros ──────────────────────────────────────────────────────────
import random, numpy as np

EPOCHS = 15
LR     = 1e-4
SEED   = 42
MIX_SEED = 29   # seed específico para synthetic_only_mix

if DEVICE.type == 'cuda':
    BATCH_SIZE  = 64 if VRAM_GB >= 40 else 32
    NUM_WORKERS = 2
elif DEVICE.type == 'mps':
    BATCH_SIZE, NUM_WORKERS = 16, 0
else:
    BATCH_SIZE, NUM_WORKERS = 8, 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(SEED)
print(f'EPOCHS={EPOCHS}  LR={LR}  BATCH={BATCH_SIZE}  MIX_SEED={MIX_SEED}')

In [ ]:
# ── Dataset, transforms y utilidades ─────────────────────────────────────────
import pandas as pd, time
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import cv2

TRAIN_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
EVAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class FlatDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label
    def class_weights(self):
        labels = np.array([l for _,l in self.samples])
        counts = np.bincount(labels)
        w = 1.0 / counts.astype(float)
        return torch.tensor([w[l] for _,l in self.samples], dtype=torch.float)

def make_loader(samples, transform, weighted=False):
    ds  = FlatDataset(samples, transform)
    pin = DEVICE.type == 'cuda'
    if weighted:
        sampler = WeightedRandomSampler(ds.class_weights(), len(ds), replacement=True)
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                         num_workers=NUM_WORKERS, pin_memory=pin)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=pin)

def resolve_path(rel_path):
    if IN_COLAB: return IMAGES_DIR / Path(rel_path).name
    return Path.cwd() / rel_path

def load_splits():
    def read(csv_path, label_filter=None):
        df = pd.read_csv(csv_path)
        if label_filter is not None: df = df[df['label'] == label_filter]
        return [(resolve_path(row['image_path']), int(row['label'])) for _,row in df.iterrows()]
    nv  = read(SPLITS_DIR/'train.csv', 0)
    mel = read(SPLITS_DIR/'train.csv', 1)
    val = read(SPLITS_DIR/'val.csv')
    tst = read(SPLITS_DIR/'test.csv')
    print(f'train nv:{len(nv)} mel:{len(mel)} | val:{len(val)} | test:{len(tst)}')
    return nv, mel, val, tst

def synth_samples_from(paths, n=None, seed=SEED):
    paths = list(paths); random.Random(seed).shuffle(paths)
    if n is not None: paths = paths[:n]
    return [(p, 1) for p in paths]

print('Utilidades listas')

In [ ]:
# ── Loop de entrenamiento con resume ─────────────────────────────────────────
import timm, torch.nn as nn
from datetime import datetime, timezone
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
from sklearn.metrics import (f1_score, roc_auc_score, recall_score,
    classification_report, confusion_matrix, roc_curve, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

# Compatibilidad PyTorch 2.6+
try:
    import numpy as _np
    torch.serialization.add_safe_globals([_np._core.multiarray.scalar])
except AttributeError:
    pass

def build_model():
    return timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)

def _eval_loop(model, loader, criterion):
    model.eval()
    loss_sum, total = 0.0, 0
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for imgs, lbs in loader:
            imgs, lbs = imgs.to(DEVICE), lbs.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, lbs)
            p      = torch.softmax(logits,1)[:,1]
            loss_sum += loss.item()*len(lbs); total += len(lbs)
            preds.extend(logits.argmax(1).cpu().tolist())
            labels.extend(lbs.cpu().tolist())
            probs.extend(p.cpu().tolist())
    return {'loss': round(loss_sum/total,4),
            'f1_mel': round(f1_score(labels,preds,pos_label=1,zero_division=0),4),
            'recall_mel': round(recall_score(labels,preds,pos_label=1,zero_division=0),4),
            'auc': round(roc_auc_score(labels,probs),4)}

def run_scenario(name, train_samples, val_samples, test_samples):
    pointer = EXP_ROOT / f'{name}_current.txt'
    run_id  = pointer.read_text().strip() if pointer.exists() \
              else f'{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}_{name}'
    if not pointer.exists(): pointer.write_text(run_id)

    run_dir     = EXP_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    done_marker = run_dir / 'test_metrics.json'
    ckpt_path   = run_dir / 'checkpoint_last.pt'

    if done_marker.exists():
        metrics = json.loads(done_marker.read_text())
        pointer.unlink(missing_ok=True)
        print(f'[{name}] Ya completado — AUC={metrics["auc"]}  Recall={metrics["recall_mel"]}  F1={metrics["f1_mel"]}')
        return metrics

    n_nv  = sum(1 for _,l in train_samples if l==0)
    n_mel = sum(1 for _,l in train_samples if l==1)
    print(f'\n{"="*60}\nEscenario: {name}\ntrain nv={n_nv} mel={n_mel} | val={len(val_samples)} test={len(test_samples)}\n{"="*60}')

    if not (run_dir/'config.json').exists():
        (run_dir/'config.json').write_text(json.dumps({
            'run_id':run_id,'scenario':name,'model':'efficientnet_b0',
            'pretrained':True,'epochs':EPOCHS,'batch_size':BATCH_SIZE,
            'lr':LR,'seed':SEED,'train_nv':n_nv,'train_mel':n_mel,
            'started_at':datetime.now(timezone.utc).isoformat()},indent=2))

    train_loader = make_loader(train_samples, TRAIN_TF, weighted=True)
    val_loader   = make_loader(val_samples,   EVAL_TF)
    test_loader  = make_loader(test_samples,  EVAL_TF)

    model     = build_model().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

    start_epoch, best_f1, history = 1, 0.0, []
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1; best_f1 = ckpt['best_f1']; history = ckpt['history']
        print(f'Retomando desde epoch {start_epoch}')

    for epoch in range(start_epoch, EPOCHS+1):
        t0 = time.time()
        model.train()
        tr_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f'[{name}] {epoch:02d}/{EPOCHS}', leave=False)
        for imgs, lbs in pbar:
            imgs, lbs = imgs.to(DEVICE), lbs.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs); loss = criterion(logits, lbs)
            loss.backward(); optimizer.step()
            tr_loss += loss.item()*len(lbs); correct += (logits.argmax(1)==lbs).sum().item(); total += len(lbs)
            pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.3f}')
        scheduler.step()
        val_m = _eval_loop(model, val_loader, criterion)
        elapsed = time.time()-t0
        row = {'epoch':epoch,'train_loss':round(tr_loss/total,4),'train_acc':round(correct/total,4),
               **{f'val_{k}':v for k,v in val_m.items()},'elapsed_s':round(elapsed,1)}
        history.append(row)
        print(f'  {epoch:02d}/{EPOCHS} | loss={tr_loss/total:.4f} | val_f1={val_m["f1_mel"]:.3f} val_auc={val_m["auc"]:.3f} | {elapsed:.0f}s')
        if val_m['f1_mel'] >= best_f1:
            best_f1 = val_m['f1_mel']; torch.save(model.state_dict(), run_dir/'best_model.pt')
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                    'scheduler':scheduler.state_dict(),'best_f1':best_f1,'history':history}, ckpt_path)

    (run_dir/'history.json').write_text(json.dumps(history, indent=2))
    model.load_state_dict(torch.load(run_dir/'best_model.pt', map_location=DEVICE, weights_only=False))
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            logits = model(imgs.to(DEVICE)); p = torch.softmax(logits,1)[:,1]
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(lbs.tolist()); all_probs.extend(p.cpu().tolist())

    report = classification_report(all_labels,all_preds,target_names=['nv','mel'],output_dict=True)
    auc    = roc_auc_score(all_labels, all_probs)
    metrics = {'auc':round(auc,4),'f1_mel':round(report['mel']['f1-score'],4),
               'recall_mel':round(report['mel']['recall'],4),
               'precision_mel':round(report['mel']['precision'],4),
               'f1_nv':round(report['nv']['f1-score'],4),'accuracy':round(report['accuracy'],4)}
    done_marker.write_text(json.dumps(metrics, indent=2))

    cfg = json.loads((run_dir/'config.json').read_text())
    cfg.update({'finished_at':datetime.now(timezone.utc).isoformat(),'test_metrics':metrics})
    (run_dir/'config.json').write_text(json.dumps(cfg, indent=2))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm, display_labels=['nv','mel']).plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}'); fig.tight_layout()
    fig.savefig(run_dir/'confusion_matrix.png', dpi=150); plt.close(fig)

    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    fig, ax = plt.subplots(figsize=(5,4))
    ax.plot(fpr, tpr, label=f'AUC={auc:.3f}'); ax.plot([0,1],[0,1],'k--',lw=0.8)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title(f'ROC — {name}'); ax.legend()
    fig.tight_layout(); fig.savefig(run_dir/'roc_curve.png', dpi=150); plt.close(fig)

    ckpt_path.unlink(missing_ok=True); pointer.unlink(missing_ok=True)
    print(f'\n[{name}] COMPLETADO  AUC={metrics["auc"]}  Recall={metrics["recall_mel"]}  F1={metrics["f1_mel"]}')
    return metrics

print('Loop de entrenamiento listo')

In [ ]:
# ── Cargar splits y fuentes ───────────────────────────────────────────────────
train_nv, train_mel, val_samples, test_samples = load_splits()

n_nv  = len(train_nv)   # 4675 — cuántos necesitamos para balancear
n_2x  = len(train_mel)  # 801

lora_paths    = sorted(LORA_DIR.glob('*.jpg'))       if LORA_DIR.exists()      else []
gan_paths     = sorted(GAN_FINAL_DIR.glob('*.png'))  if GAN_FINAL_DIR.exists() else []

print(f'LoRA:      {len(lora_paths)} imágenes')
print(f'GAN-final: {len(gan_paths)} imágenes')
print(f'n_2x={n_2x}  n_nv={n_nv} (target para balanced)')

## Escenario 1 — `synthetic_only_lora`
Solo LoRA, sin imágenes reales de melanoma. Compara directamente con `synthetic_only` (TI, AUC=0.686).

In [ ]:
results_sol = run_scenario(
    name          = 'synthetic_only_lora',
    train_samples = train_nv + synth_samples_from(lora_paths, n=n_2x),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 2 — `synthetic_only_gan`
Solo GAN-final, sin imágenes reales de melanoma.

In [ ]:
results_sog = run_scenario(
    name          = 'synthetic_only_gan',
    train_samples = train_nv + synth_samples_from(gan_paths, n=n_2x),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 3 — `synthetic_only_mix`
401 LoRA + 400 GAN-final (seed=29), sin imágenes reales. Mezcla equilibrada de ambas fuentes.

In [ ]:
mix_lora = synth_samples_from(lora_paths, n=401, seed=MIX_SEED)
mix_gan  = synth_samples_from(gan_paths,  n=400, seed=MIX_SEED)

results_som = run_scenario(
    name          = 'synthetic_only_mix',
    train_samples = train_nv + mix_lora + mix_gan,
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 4 — `real_balanced_lora`
801 real + ~3874 LoRA para igualar nv≈mel. Compara con `real_balanced` (AUC=0.9257, TI mezclada).

In [ ]:
n_lora_needed = n_nv - n_2x  # ~3874

results_rbl = run_scenario(
    name          = 'real_balanced_lora',
    train_samples = train_nv + train_mel + synth_samples_from(lora_paths, n=n_lora_needed),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 5 — `real_balanced_gan`
801 real + ~3874 GAN-final para igualar nv≈mel.

In [ ]:
n_gan_needed = n_nv - n_2x  # ~3874

results_rbg = run_scenario(
    name          = 'real_balanced_gan',
    train_samples = train_nv + train_mel + synth_samples_from(gan_paths, n=n_gan_needed),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Comparación final — todos los experimentos

In [ ]:
SCENARIO_ORDER = [
    'real_only', 'real_balanced', 'real_2x', 'synthetic_only',
    'img2img_2x', 'ti_filtered_2x', 'gan64_2x', 'img2img_all', 'img2img_only',
    'gan_final_2x', 'lora_2x',
    'synthetic_only_lora', 'synthetic_only_gan', 'synthetic_only_mix',
    'real_balanced_lora', 'real_balanced_gan',
]

all_results = {}
for sc in SCENARIO_ORDER:
    run_dir = find_run_dir(sc)
    if run_dir and (run_dir/'test_metrics.json').exists():
        m   = json.loads((run_dir/'test_metrics.json').read_text())
        cfg = json.loads((run_dir/'config.json').read_text()) if (run_dir/'config.json').exists() else {}
        all_results[sc] = {'metrics': m, 'config': cfg}
    else:
        print(f'  {sc}: no completado — omitido')

if all_results:
    rows = []
    for sc, data in all_results.items():
        m, cfg = data['metrics'], data['config']
        rows.append({'Escenario':sc, 'Train mel':cfg.get('train_mel','?'),
                     'AUC':m['auc'], 'Recall mel':m['recall_mel'],
                     'F1 mel':m['f1_mel'], 'Precision mel':m['precision_mel'],
                     'Accuracy':m['accuracy']})
    df = pd.DataFrame(rows).set_index('Escenario')
    print(df.to_string())
    out = EXP_ROOT / 'comparison_final.json'
    out.write_text(json.dumps({sc: d['metrics'] for sc,d in all_results.items()}, indent=2))
    print(f'\nGuardado en {out}')

In [ ]:
if len(all_results) >= 2:
    # Grupos de color
    baseline = {'real_only','real_balanced','real_2x','synthetic_only'}
    ronda1   = {'img2img_2x','ti_filtered_2x','gan64_2x','img2img_all','img2img_only','gan_final_2x','lora_2x'}
    ronda2   = {'synthetic_only_lora','synthetic_only_gan','synthetic_only_mix','real_balanced_lora','real_balanced_gan'}
    palette  = {sc: ('#aec6e8' if sc in baseline else '#4C72B0' if sc in ronda1 else '#C44E52') for sc in all_results}

    sc_list  = list(all_results.keys())
    metrics_k = [('auc','AUC'),('recall_mel','Recall Melanoma'),('f1_mel','F1 Melanoma')]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (mk, title) in zip(axes, metrics_k):
        vals  = [all_results[sc]['metrics'][mk] for sc in sc_list]
        cols  = [palette[sc] for sc in sc_list]
        bars  = ax.bar([s.replace('_','\n') for s in sc_list], vals, color=cols)
        ax.set_ylim(0, 1.08); ax.set_title(title, fontsize=11)
        ax.tick_params(axis='x', labelsize=7)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)
        if 'real_only' in all_results:
            bl = all_results['real_only']['metrics'][mk]
            ax.axhline(bl, color='red', linestyle='--', lw=0.8, label='real_only')
            ax.legend(fontsize=8)

    from matplotlib.patches import Patch
    fig.legend(handles=[
        Patch(facecolor='#aec6e8', label='Baseline'),
        Patch(facecolor='#4C72B0', label='Ronda 1 (fuentes aisladas)'),
        Patch(facecolor='#C44E52', label='Ronda 2 (synthetic-only / balanced)'),
    ], loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5,-0.08))
    fig.suptitle('HAM10000 — Comparación completa de escenarios', fontsize=13)
    fig.tight_layout()
    out = EXP_ROOT / 'comparison_final_plot.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot guardado en {out}')